<a href="https://colab.research.google.com/github/Symbiosis-Quantum-Club/Quantum-Computing-and-QML-a-Comprehensive-Roadmap/blob/main/1__Intro_to_Cirq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Fix dependency conflicts by installing a compatible stack
!pip uninstall -y numpy scipy cirq
!pip install --quiet "numpy<2.0.0" "scipy<1.14.0" "cirq==1.4.1"

# 2. Force a runtime restart to load the correct NumPy version
# After this cell runs, the kernel will restart automatically.
import os
os._exit(0)

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: cirq 1.4.1
Uninstalling cirq-1.4.1:
  Successfully uninstalled cirq-1.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
jaxlib 0.7.2 requ

## 1. Defining Quantum Hardware (Qubits)
In Cirq, you don't just 'create a qubit'; you define where it lives on a device. This reflects the physical reality of quantum hardware, where qubits are arranged in specific topologies (like lines or grids).

* **LineQubits**: Arranged in a 1D chain.
* **GridQubits**: Arranged in a 2D lattice (common for Google's Sycamore processor).
* **NamedQubits**: Useful for abstract labels or ancilla qubits.

# Introduction to Cirq Quantum Framework for New Learners

This notebook provides an overview of **Cirq**, Google's open-source quantum computing library. It aims to introduce core concepts, demonstrate basic usage, and highlight key differences from other popular frameworks like Qiskit.

Cirq is designed for writing, manipulating, and optimizing quantum circuits, and running them on quantum computers or simulators. It emphasizes precise control over quantum operations and is particularly well-suited for near-term (NISQ) quantum devices.

### Further Learning Resources:
- [Cirq Official Documentation](https://quantumai.google/cirq/)
- [Cirq Tutorials](https://quantumai.google/cirq/tutorials)
- [Google Quantum AI Blog](https://quantumai.google/)

In [1]:
import cirq
import numpy as np

print("--- 1. Qubits in Cirq ---")
# Verify versions
print(f"Cirq version: {cirq.__version__}")
print(f"NumPy version: {np.__version__}\n")

# Defining Qubits
q0 = cirq.LineQubit(0)
q1 = cirq.LineQubit(1)
q_ancilla = cirq.NamedQubit('ancilla')

print(f"Line Qubits: {q0}, {q1}")
print(f"Named Qubit: {q_ancilla}")

--- 1. Qubits in Cirq ---
Cirq version: 1.4.1
NumPy version: 1.26.4

Line Qubits: q(0), q(1)
Named Qubit: ancilla


## 2. Building a Quantum Circuit (Moments vs. Gates)

A defining feature of Cirq is the **Moment**. While other frameworks like Qiskit add gates sequentially, Cirq groups operations into Moments. A Moment is a collection of operations that happen at the same slice of time.

This provides explicit control over the temporal structure of your circuit. This is vital for hardware optimization, as it allows you to precisely manage when gates are applied and when qubits are idle.

In [2]:
# Create an empty circuit
circuit = cirq.Circuit()

# Moment 1: Apply a Hadamard gate to q0
circuit.append(cirq.H(q0))

# Moment 2: Apply a CNOT gate between q0 and q1
circuit.append(cirq.CNOT(q0, q1))

# Moment 3: Apply parallel gates to q1 and q_ancilla
# Cirq automatically groups these into a single 'Moment'
circuit.append([cirq.X(q1), cirq.Z(q_ancilla)])

# Moment 4: Measurement
circuit.append(cirq.measure(q0, q1, key='result'))

print("Visualizing the Circuit:")
print(circuit)

print("\nCircuit Moments:")
for i, moment in enumerate(circuit):
    print(f"Moment {i}: {moment}")

Visualizing the Circuit:
0: ─────────H───@───────M('result')───
                │       │
1: ─────────────X───X───M─────────────

ancilla: ───Z─────────────────────────

Circuit Moments:
Moment 0:         ╷ None 0
╶───────┼────────
ancilla │ Z
        │
0       │      H
        │
Moment 1:   ╷ 0 1
╶─┼─────
0 │ @─X
  │
Moment 2:   ╷ 1
╶─┼───
0 │ X
  │
Moment 3:   ╷ 0           1
╶─┼───────────────
0 │ M('result')─M
  │


### Understanding the Diagram
In the visualization above:
* The horizontal lines represent the 'world line' or timeline of each qubit.
* The vertical alignment shows **Moments**. Gates in the same vertical column are intended to run at the same time.
* Notice that Cirq automatically tries to push gates to the earliest possible Moment (the 'left') unless you specify a different insertion strategy.

## 3. Simulating the Circuit

Cirq comes with a built-in `Simulator` that can execute circuits on your local machine. By default, the simulator uses a wavefunction-based approach to provide sampling results.

In [3]:
# Initialize the simulator
simulator = cirq.Simulator()

# Run the circuit for a specific number of repetitions
# Note: Measurement keys are used to retrieve specific results
result = simulator.run(circuit, repetitions=1000)

# Print the results
print("Measurement results (first 10 samples):")
print(result.measurements['result'][:10])

# Generate and plot a histogram
print("\nOutcome Histogram:")
print(result.histogram(key='result'))

Measurement results (first 10 samples):
[[1 0]
 [0 1]
 [0 1]
 [1 0]
 [0 1]
 [1 0]
 [1 0]
 [1 0]
 [1 0]
 [1 0]]

Outcome Histogram:
Counter({2: 509, 1: 491})


## 4. Beyond Sampling: The State Vector
When learning or debugging, we often want to see the exact probability amplitudes of the qubits before they are collapsed by measurement.

Cirq's `simulate()` method is distinct from `run()`. While `run()` mimics physical hardware by returning bitstrings (0s and 1s), `simulate()` provides access to the underlying mathematical state (the state vector).

In [4]:
# To see the state vector, we simulate a circuit without terminal measurements
# Creating a simple Bell State circuit for clarity
bell_circuit = cirq.Circuit(cirq.H(q0), cirq.CNOT(q0, q1))

sim = cirq.Simulator()
sv_result = sim.simulate(bell_circuit)

print("Final State Vector:")
print(sv_result.dirac_notation())
print("\nFull Vector:")
print(sv_result.state_vector())

Final State Vector:
0.71|00⟩ + 0.71|11⟩

Full Vector:
[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


### Quick Transition Guide: Qiskit to Cirq
| Feature | Qiskit | Cirq |
| :--- | :--- | :--- |
| **Qubit ID** | Index (0, 1...) | Location (Line, Grid, Name) |
| **Circuit** | `QuantumCircuit` | `Circuit` |
| **Execution** | `backend.run()` | `simulator.run()` |
| **Parallelism** | Automatic (Transpiler) | Explicit (Moments) |

### Summary
- **Qubits**: Defined by location (Line, Grid, Named).
- **Moments**: Explicit time slices for parallel operations.
- **Simulator**: Local execution for verification and testing.

This notebook is now fully functional and provides a robust foundation for learning Cirq.